<a href="https://colab.research.google.com/github/jwalkorat/Agri-Vision/blob/main/model/unet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn

import torchvision
import torchvision.transforms.functional as TF
import os

from typing import Tuple, Dict, List

In [ ]:
class DoubleConv(nn.Module):
  def __init__(self, in_features, out_features):
    super().__init__()

    self.conv_block_1 = nn.Sequential(
        nn.Conv2d(in_channels=in_features,
                  out_channels=out_features,
                  kernel_size=3,
                  padding=1,
                  stride=1,
                  bias=False),
        nn.BatchNorm2d(num_features=out_features),
        nn.ReLU(),

        nn.Conv2d(in_channels=out_features,
                  out_channels=out_features,
                  kernel_size=3,
                  padding=1,
                  stride=1,
                  bias=False),
        nn.BatchNorm2d(num_features=out_features),
        nn.ReLU()
    )

  def forward(self, x):
    return self.conv_block_1(x)

In [ ]:
class UNET(nn.Module):
  def __init__(self, in_features=3, out_features=1, features=[64, 128, 256, 512]):
    super().__init__()

    self.ups = nn.ModuleList()
    self.downs = nn.ModuleList()

    self.pool = nn.MaxPool2d(kernel_size=2)

    for feature in features:
      self.downs.append(DoubleConv(
          in_features=in_features,
          out_features=feature
      ))
      in_features = feature

    for feature in reversed(features):
      self.ups.append(
          nn.ConvTranspose2d(
              in_channels=2 * feature,
              out_channels=feature,
              kernel_size=2,
              stride=2
          )
      )
      self.ups.append(DoubleConv(
          in_features=2 * feature,
          out_features=feature
      ))

    self.bottleneck = DoubleConv(in_features=features[-1],
                                 out_features=2 * features[-1])

    self.fc = nn.Conv2d(
        in_channels=features[0],
        out_channels=out_features,
        kernel_size=1,
        padding=0,
        stride=1
    )

  def forward(self, x):
    skip_connections = []

    for down in self.downs:
      x = down(x)
      skip_connections.append(x)
      x = self.pool(x)

    x = self.bottleneck(x)
    skip_connections = skip_connections[::-1]

    for idx in range(0, len(self.ups), 2):
      transposed = self.ups[idx](x)
      skip_concat = torch.cat((transposed, skip_connections[idx // 2]), dim=1)
      x = self.ups[idx + 1](skip_concat)

    return self.fc(x)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

model = UNET().to(device)

train_batch = torch.rand((1, 3, 512, 512)).to(device)

with torch.inference_mode():
  preds = model(train_batch)

preds.shape

torch.Size([1, 1, 512, 512])